In [ ]:
%load_ext autoreload
%autoreload 2
%cd /my_dir/factowl/factowl/
!pip install -e ./
!pip -q install wikipedia jieba
%cd factowl/

In [ ]:
%cd factowl/

In [10]:
import json
import os
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from vllm import LLM, SamplingParams
#from transformers import AutoTokenizer
from factowl.io import save_eval_results, save_predictions, load_json_generations
from factowl.factscorer_sped_up_vllm import FactScorerSpedUpVLLM as FactScorer

In [11]:
hf_token = 'hf_token'
os.environ["CUDA_VISIBLE_DEVICES"]="0,1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# model_name = 'meta-llama/Meta-Llama-3-8B-Instruct'
# model_name="Qwen/Qwen2.5-7B-Instruct"
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
cache_dir = './.cache/'

GEN_COLS = ('llama3.1:8b', 'qwen2.5:7b', )
domains = ["cars", "rivers", "bad_weather"]

cnp = 1 # Number of context pages retrieved from Wikipedia API.
nsp = 5 # Number of relevant passages retrieved to support a single atomic fact.
context_type = 'wikipedia_api'
SETUP = "retrieval+llama"

In [13]:
def load_json_data(p, gen_cols):
    topics = []
    gens = {col: [] for col in gen_cols}
    popularities = []
    bad_topics = []
    ctxs = []

    with open(p, 'r', encoding="utf-8") as f:
        docs = json.load(f)
        print(len(docs))
        for d in tqdm(docs):
            t = d["title_en"].strip('.')
            gens_dict = {col: d[col] for col in gen_cols}
            p = d["popularity_part_sector"]
            topics.append(t)
            popularities.append(p)
  
            for col, gen_str in gens_dict.items():
                gens[col].append(gen_str)
    return topics, gens, popularities

def create_topic2cxt_from_json(p,  topic_col="title_en", cxt_col="wikipedia_page_en"):
    topic2cxt = {}
    with open(p, 'r') as f:
        docs = json.load(f)
        for doc in docs:
            topic = doc[topic_col]
            ctx = doc[cxt_col]
            topic2cxt[topic] = ctx.split("\n\n\n")
    return topic2cxt

In [ ]:

vllm_model = LLM(
    model=model_name,
    gpu_memory_utilization=0.8,
    # trust_remote_code=True,
)

In [16]:
POP_MAP = {
0: "high popularity",
1: "modest populrity",
2: "low popularity"
}

def aggregate_results(topics, generations, popularities, out):
    topic2decisions = {}
    
    for ad in out["decisions"]:
        t = ad["topic"]
        if topic2decisions.get(t) is None:
            topic2decisions[t] = []
        topic2decisions[t].append(ad)
 
    all_true, low_true, med_true, high_true = 0,0,0,0
    all_total, low_total, med_total, high_total = 0,0,0,0
    for t, p in zip(topics, popularities):
        p = int(p)
        if topic2decisions.get(t) is None:
            continue
        decs = topic2decisions[t]
        
        for dec in decs:
            is_sup = dec["is_supported"]
            all_total += 1
            if p == 0:
                high_total += 1
            elif p == 1:
                med_total += 1
            elif p == 2:
                low_total += 1
            else:
                raise Exception(f"Invalid popularity: {p}")
            if is_sup == True:
                all_true += 1
                if p == 0:
                    high_true += 1
                elif p == 1:
                    med_true += 1
                elif p == 2:
                    low_true += 1
    print(f"Calculated precision:")
    print(f"Total: {all_true / all_total} ({all_true} / {all_total})")
    print(f"Low: {low_true / low_total} ({low_true} / {low_total})")
    print(f"Medium: {med_true / med_total} ({med_true} / {med_total})")
    print(f"High: {high_true / high_total} ({high_true} / {high_total})")
    

In [ ]:
for domain in domains:
   for gen_col in GEN_COLS:
      p=f"./data/1000_{domain}_with_cleared_refs.json"
      p1=f"./data/1000_{domain}_with_llm_gen_en.json"

      topics, generations_dict, popularities = load_json_data(p1, GEN_COLS)
      topic2ctx = create_topic2cxt_from_json(p)
      generations = generations_dict[gen_col]
      
      print(f"Domain: {domain}")
      print(f"Generation column: {gen_col}")      
      print(f"Evaluation for {len(topics)} generations")
        
      base_dir = "./"
      atomic_facts_cache_dir = f"cache-ridic-sep/en-llama-filtered/{domain}-{gen_col}-{SETUP}/"
        
      fs = FactScorer(model_name=SETUP,
                  #  data_dir=data_dir,
                   vllm_model=vllm_model,
                   dump_every_int=100,
                   cache_dir=cache_dir,
                   abstain_detection_type="generic",
                   use_this_topic2content_only=topic2ctx,
                   lang="en",
                   debug=True)
        
      out = fs.get_score(topics, generations, gamma=10, knowledge_source="enwiki-20230401", verbose=True)
        
      p = f"./pred_{domain}-{gen_col}_{SETUP}_eval-{context_type}-p{cnp}-c{nsp}.tsv"
      save_predictions(out, p, print_res=False)
        
      p = f"./eval_{domain}-{gen_col}_res_{SETUP}_eval-{context_type}-p{cnp}-c{nsp}.tsv"
      save_eval_results(out, p)
        
      aggregate_results(topics, generations, popularities, out)